# GCAP3226 Week 3 — Session 1
## Decision rule + one slow MSW example

**Follow along** with the instructor (not graded).  
**Dataset:** `GCAP3226_week3.csv` (same folder as this notebook).  
**On the projector:** policy hook + **survey question wording** (see PPT).

### Today's aims
1. Know **when regression helps** a policy question (and when it does not).
2. Map survey items to **Y** and **X**.
3. Run **one** simple linear regression path and practise cautious interpretation.


## 0. Decision rule (read with the PPT)

**Regression is a good fit when** you have:
- many **observed cases** (people, districts, facilities…);
- a **clear outcome** you care about (Y);
- a **small set of predictors** (X) already chosen for the question;
- a need for **association / adjusted comparison** (“how does Y move with X?”).

**Prefer another tool (e.g. Week 4 simulation) when** the core question is “what if we change capacity, arrivals, or a queue?” rather than “what patterns appear in observed survey/admin data?”

**Hard rule for this course:** association ≠ causation. A coefficient is not a licence to tell the public that X *causes* Y.


## 1. Survey items → variables (align with PPT)

Exact wording is on the slides. In the CSV we use these column names:

| Role | Column | Typical meaning (Likert 1–5 unless noted) |
|------|--------|-------------------------------------------|
| **Y (outcome)** | `support_info` | Support / willingness toward MSW charging after information |
| **X1** | `government_consideration` | Perception that government considers public views |
| **X2** | `fairness` | Perception that the policy is fair |
| Context | sample | Convenience / non-probability (n ≈ 97) — limits generalisation |

We will treat `support_info` as **approximately continuous** for a teaching linear model (same practice as last year). That is a modelling choice, not a claim that attitudes are truly continuous.


## 2. Plumbing — imports and load

Run the next cells. Raise your hand only if import or file load fails (quick green-light; Week 2 already covered the environment).


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Load data from the same folder as this notebook
df = pd.read_csv(".../GCAP3226_week3.csv")
print("shape:", df.shape)
df[["support_info", "government_consideration", "fairness"]].head()


FileNotFoundError: [Errno 2] No such file or directory: '.../GCAP3226_week3.csv'

In [ ]:
# Quick look at the teaching variables
cols = ["support_info", "government_consideration", "fairness"]
print(df[cols].describe().round(2))
print("\nMissing values:")
print(df[cols].isna().sum())


## 3. One slow path — simple linear regression

**Policy-tinged question (working version):**  
Among surveyed respondents, how is perceived **government consideration** associated with **support** for MSW charging?

We fit:

\[ \widehat{support\_info} = \hat\beta_0 + \hat\beta_1 \, government\_consideration \]

Instructor pace: run → read the coefficient → look at the scatter → pause for discussion.


In [ ]:
# Drop rows missing Y or X for this simple model
simple = df[["support_info", "government_consideration"]].dropna()
X = simple[["government_consideration"]]
y = simple["support_info"]

model = LinearRegression()
model.fit(X, y)
y_hat = model.predict(X)

beta0 = model.intercept_
beta1 = model.coef_[0]
r2 = r2_score(y, y_hat)

print(f"Intercept (beta0): {beta0:.3f}")
print(f"Slope (beta1) for government_consideration: {beta1:.3f}")
print(f"R-squared: {r2:.3f}")
print(f"n used: {len(simple)}")


In [ ]:
# Sanity plot: data + fitted line
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(simple["government_consideration"], simple["support_info"], alpha=0.7, label="respondents")
x_line = np.linspace(simple["government_consideration"].min(), simple["government_consideration"].max(), 50)
ax.plot(x_line, beta0 + beta1 * x_line, color="C1", label="fitted line")
ax.set_xlabel("government_consideration (survey X)")
ax.set_ylabel("support_info (survey Y)")
ax.set_title("MSW support vs perceived government consideration")
ax.legend()
plt.tight_layout()
plt.show()


### Pause prompts (discuss aloud; optional notes below)

1. In **plain language**, what is Y in this notebook?  
2. The slope `beta1` is positive / negative / near zero — what does that **suggest** (association only)?  
3. What would you **still refuse to claim** to a district audience after seeing this plot?


In [ ]:
# Optional: type short notes from the class discussion
# 1. Y in plain language:
# 2. What the slope suggests:
# 3. What we still cannot claim:


## 4. Optional stretch (only if time) — add fairness

Same Y; two predictors. Still **not** causal. Still the same non-probability sample.


In [ ]:
multi = df[["support_info", "government_consideration", "fairness"]].dropna()
X2 = multi[["government_consideration", "fairness"]]
y2 = multi["support_info"]

model2 = LinearRegression()
model2.fit(X2, y2)

print("Intercept:", round(model2.intercept_, 3))
for name, coef in zip(X2.columns, model2.coef_):
    print(f"coef[{name}]: {coef:.3f}")
print("R-squared:", round(r2_score(y2, model2.predict(X2)), 3))


## Bridge to Session 2

Next session you will **not** learn a new model family.  
You will re-use this same run in pairs: translate the slope into careful policy language, and write one line you would **not** say in public.
